# Colab 1 - CrewAI Supply-Chain Pipeline
### Day 16: Multi-Agent Coordination Patterns  |  Powered by Groq

**Scenario:** GlobalFlow Logistics moves 4M parcels/day across 38 countries.
A 2-hour port delay costs EUR 14M. Build a 5-agent crew that detects disruptions
and coordinates a full response automatically.

**What you will build:**
- 5 specialised agents (Monitor, Router, Comms, Compliance, Reporter)
- Task dependency chain with `context=` injection
- Hierarchical process orchestrated by a Groq manager LLM
- Long-term memory across crew runs
- Executive disruption report saved to disk

**LLM:** [Groq](https://console.groq.com) - free tier, fast inference
**Models:** `llama-3.3-70b-versatile` (reasoning) | `llama-3.1-8b-instant` (simple tasks)

> Get a free Groq API key at https://console.groq.com/keys - no credit card required.

**Time budget:** ~85 min core + 30 min extension tasks

## Part 1 - Environment Setup (15 min)

In [6]:
# Cell 1 - Install dependencies
# crewai[litellm] enables Groq support via LiteLLM bridge (required for crewai v1+)

!pip install "crewai[litellm]" crewai-tools langchain-groq -q

print("[OK] Packages installed:")
print("  crewai[litellm] - CrewAI with Groq via LiteLLM")
print("  crewai-tools    - FileWriterTool, SerperDevTool, etc.")
print("  langchain-groq  - ChatGroq for LangGraph (used in Colab 2)")

[OK] Packages installed:
  crewai[litellm] - CrewAI with Groq via LiteLLM
  crewai-tools    - FileWriterTool, SerperDevTool, etc.
  langchain-groq  - ChatGroq for LangGraph (used in Colab 2)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.


In [7]:
# Cell 2 - Configure Groq API key
import os

# Option A: Colab Secrets (recommended)
# Add GROQ_API_KEY in the lock icon panel on the left sidebar
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("[OK] GROQ_API_KEY loaded from Colab Secrets")
except Exception:
    # Option B: paste directly (do not share the notebook with key visible)
    os.environ["GROQ_API_KEY"] = "gsk_..."  # <- replace with your key
    print("[WARN] Using hardcoded key - switch to Colab Secrets for production.")

key = os.environ.get("GROQ_API_KEY", "")
if key and key != "gsk_...":
    print(f"  Key prefix: {key[:8]}...")
else:
    print("[NO] GROQ_API_KEY not set - cells below will fail until you set it.")

[WARN] Using hardcoded key - switch to Colab Secrets for production.
[NO] GROQ_API_KEY not set - cells below will fail until you set it.


In [8]:
# Fix for Windows: install pywin32 first, then reinstall crewai cleanly
!pip install pywin32 -q
!pip install "crewai[litellm]" crewai-tools langchain-groq --force-reinstall -q

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
E

In [9]:
# Cell 3 - Smoke test: single-agent hello world via Groq
from crewai import Agent, Task, Crew, Process

# In CrewAI v1, Groq models are referenced as 'groq/<model_name>'
# LiteLLM handles the Groq API call under the hood.
GROQ_FAST    = "groq/llama-3.1-8b-instant"       # fast + cheap, good for simple tasks
GROQ_SMART   = "groq/llama-3.3-70b-versatile"    # capable, good for reasoning
GROQ_MANAGER = "groq/llama-3.3-70b-versatile"    # hierarchical manager LLM

test_agent = Agent(
    role="Hello World Agent",
    goal="Confirm the CrewAI + Groq environment is working correctly",
    backstory="A simple validation agent. You respond in exactly one sentence.",
    llm=GROQ_FAST,
    verbose=False,
    max_iter=2,
)

test_task = Task(
    description="Confirm you are running on Groq and identify your model in one sentence.",
    expected_output="One sentence confirming the environment works.",
    agent=test_agent,
)

test_crew = Crew(agents=[test_agent], tasks=[test_task], verbose=False)
result = test_crew.kickoff()

print("[OK] Smoke test PASSED - CrewAI + Groq is working")
print()
print(result.raw)

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

BadRequestError: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}


## Part 2 - Define 5 GlobalFlow Agents (40 min)

In [ ]:
# Cell 4 - Tools and LLM tier constants
from crewai import Agent
from crewai_tools import FileWriterTool
from crewai.tools import BaseTool
from pydantic import Field

# LLM tiers
GROQ_FAST    = "groq/llama-3.1-8b-instant"
GROQ_SMART   = "groq/llama-3.3-70b-versatile"
GROQ_MANAGER = "groq/llama-3.3-70b-versatile"

file_writer = FileWriterTool()

# Mock search tool - no Serper API key required for this lab
# In production replace with: from crewai_tools import SerperDevTool
class MockSearchTool(BaseTool):
    name: str = "web_search"
    description: str = (
        "Search the web for supply-chain disruption news, "
        "shipping route data, and regulatory information."
    )

    def _run(self, query: str) -> str:
        return (
            f"[SIMULATED SEARCH] Results for: '{query}'\n"
            "Rotterdam: 18h closure, storm surge, severity 8/10\n"
            "Alternative 1 - Hamburg: +5h, -6% cost, low risk\n"
            "Alternative 2 - Felixstowe: +8h, -10% cost, medium risk\n"
            "Alternative 3 - Antwerp: +3h, +2% cost, low risk\n"
            "Singapore PSA: normal operations, no disruption\n"
        )

search_tool = MockSearchTool()

print("[OK] LLM tiers and tools configured")
print(f"  Fast model:    {GROQ_FAST}")
print(f"  Smart model:   {GROQ_SMART}")
print(f"  Manager model: {GROQ_MANAGER}")

In [ ]:
# Cell 5 - Agent 1: Disruption Monitor
disruption_monitor = Agent(
    role="Supply Chain Disruption Monitor",
    goal=(
        "Continuously scan for logistics disruptions - port closures, weather events, "
        "customs delays, and supplier failures - and assess their severity on a 1-10 scale."
    ),
    backstory=(
        "You are a veteran logistics intelligence analyst with 12 years at Maersk and DHL. "
        "You have seen every kind of supply-chain disruption imaginable, from Suez Canal "
        "blockages to pandemic port shutdowns. You are calm under pressure, deeply data-driven, "
        "and always quantify impact before escalating. You write in crisp bullet points."
    ),
    llm=GROQ_SMART,
    tools=[search_tool],
    verbose=True,
    max_iter=4,
)

print("[OK] Agent 1:", disruption_monitor.role)

In [ ]:
# Cell 6 - Agent 2: Route Optimiser
route_optimiser = Agent(
    role="Logistics Route Optimiser",
    goal=(
        "Given a disruption report, calculate the 3 best alternative routes for affected "
        "shipments, ranking by total cost + estimated delay. Provide a clear recommendation."
    ),
    backstory=(
        "You are a PhD-level operations research specialist who spent 8 years building "
        "real-time routing algorithms for FedEx. You think in graphs, costs, and probabilities. "
        "You know every major shipping lane, air corridor, and rail route. You always present "
        "a primary recommendation plus two ranked alternatives with a weighted score."
    ),
    llm=GROQ_SMART,
    tools=[search_tool],
    verbose=True,
    max_iter=4,
)

print("[OK] Agent 2:", route_optimiser.role)

In [ ]:
# Cell 7 - Agent 3: Supplier Communications Specialist
supplier_comms = Agent(
    role="Supplier Communications Specialist",
    goal=(
        "Draft professional, urgent communications to affected suppliers and carriers "
        "explaining the disruption, proposing alternatives, and requesting confirmation "
        "within 4 hours."
    ),
    backstory=(
        "You are a senior procurement manager who has negotiated contracts in 22 countries. "
        "You are culturally fluent, direct but diplomatic, and always frame disruptions as "
        "collaborative problems to solve, never as blame assignments. You know that tone "
        "in a crisis email can make or break a supplier relationship worth millions."
    ),
    llm=GROQ_FAST,      # Drafting emails does not need the 70B model
    verbose=True,
    max_iter=3,
)

print("[OK] Agent 3:", supplier_comms.role)

In [ ]:
# Cell 8 - Agent 4: Compliance Officer
compliance_officer = Agent(
    role="Trade Compliance Officer",
    goal=(
        "For each proposed re-route, verify customs requirements, check for sanctions or "
        "restricted-goods regulations, and flag any compliance risks. Issue a COMPLIANCE "
        "CLEARED or COMPLIANCE HOLD recommendation."
    ),
    backstory=(
        "You are a Certified Customs Specialist (CCS) with deep expertise in EU, US, and "
        "APAC trade regulations. You have worked with the WTO and have a zero-tolerance "
        "approach to compliance shortcuts. A single customs violation can cost more than "
        "the disruption itself."
    ),
    llm=GROQ_SMART,
    tools=[search_tool],
    verbose=True,
    max_iter=4,
)

print("[OK] Agent 4:", compliance_officer.role)

In [ ]:
# Cell 9 - Agent 5: Executive Report Writer
report_writer = Agent(
    role="Executive Communications Writer",
    goal=(
        "Synthesise the disruption intelligence, route options, supplier actions, and "
        "compliance status into a clear, actionable executive briefing. "
        "Format: Situation -> Impact -> Response -> Next Steps. Maximum 1 page."
    ),
    backstory=(
        "You are a former management consultant who spent 10 years writing board-level "
        "crisis communications for Fortune 500 logistics companies. You eliminate jargon "
        "ruthlessly, lead with the bottom line, and always end with exactly 3 numbered "
        "action items with named owners and deadlines."
    ),
    llm=GROQ_SMART,
    tools=[file_writer],
    verbose=True,
    max_iter=3,
)

print("[OK] Agent 5:", report_writer.role)
print()
print("All 5 GlobalFlow agents ready.")

### 2b - Define Tasks with Context Dependencies

In [ ]:
# Cell 10 - Task 1: Monitor disruptions
from crewai import Task

task_monitor = Task(
    description=(
        "Search for active logistics disruptions affecting GlobalFlow's key corridors: "
        "Rotterdam (EU hub), Singapore (APAC hub), Houston (US hub), and the AE-1 "
        "Asia-Europe shipping lane. Report: (1) disruption type and location, "
        "(2) severity score 1-10, (3) estimated duration, (4) shipments likely affected. "
        "Start your report with 'SEVERITY: X/10' on the first line."
    ),
    expected_output=(
        "Structured disruption report: severity score, affected corridors, "
        "shipment count, estimated duration, recommended escalation level."
    ),
    agent=disruption_monitor,
)

print("[OK] Task 1: Monitor disruptions  (no dependencies)")

In [ ]:
# Cell 11 - Task 2: Route optimisation (depends on Task 1)
task_route = Task(
    description=(
        "Using the disruption report in your context, calculate 3 alternative routes "
        "for the 50 highest-priority shipments. For each route: "
        "(1) route name and via-points, (2) cost delta vs standard (%), "
        "(3) delay in hours, (4) risk score 1-5, (5) CO2 delta. "
        "Rank by weighted score: 60% cost, 30% time, 10% risk."
    ),
    expected_output=(
        "Ranked table of 3 alternative routes with cost delta, delay, risk, "
        "weighted score, and a one-sentence rationale for the top choice."
    ),
    agent=route_optimiser,
    context=[task_monitor],   # <- injects task_monitor output into this task
)

print("[OK] Task 2: Route optimisation   (context: task_monitor)")

In [ ]:
# Cell 12 - Task 3: Supplier comms (depends on Tasks 1 + 2)
task_comms = Task(
    description=(
        "Draft communications to the 3 most critical affected suppliers. "
        "For each: (1) subject line, (2) 150-word email body explaining the disruption, "
        "the proposed re-routing option, and requesting confirmation within 4 hours. "
        "Tone: professional, urgent, collaborative."
    ),
    expected_output=(
        "Three complete email drafts formatted as:\n"
        "[SUPPLIER NAME] / [SUBJECT LINE]\n[EMAIL BODY]"
    ),
    agent=supplier_comms,
    context=[task_monitor, task_route],
)

print("[OK] Task 3: Supplier comms       (context: task_monitor, task_route)")

In [ ]:
# Cell 13 - Task 4: Compliance check (depends on Task 2)
task_compliance = Task(
    description=(
        "Review the top-ranked re-routing option from the route optimisation team. "
        "Check: (1) customs requirements per transit country, "
        "(2) sanctions or dual-use goods restrictions, "
        "(3) certificate of origin implications. "
        "Issue COMPLIANCE CLEARED or COMPLIANCE HOLD with detailed reasoning."
    ),
    expected_output=(
        "Compliance status (CLEARED or HOLD), per-country requirements, "
        "flags with remediation steps, estimated customs processing time."
    ),
    agent=compliance_officer,
    context=[task_route],
)

print("[OK] Task 4: Compliance check     (context: task_route)")

In [ ]:
# Cell 14 - Task 5: Executive report (all context)
task_report = Task(
    description=(
        "Compile all outputs into a single executive briefing.\n"
        "Use these exact headings:\n"
        "  SITUATION: what happened and severity\n"
        "  IMPACT: shipments affected, EUR cost exposure\n"
        "  RESPONSE: chosen re-route, supplier actions, compliance status\n"
        "  NEXT STEPS: exactly 3 numbered actions with owners and deadlines\n"
        "Maximum 400 words. Save to file 'globalflow_disruption_report.txt'."
    ),
    expected_output=(
        "Complete executive briefing saved to 'globalflow_disruption_report.txt', "
        "4-section structure, maximum 400 words."
    ),
    agent=report_writer,
    context=[task_monitor, task_route, task_comms, task_compliance],
    output_file="globalflow_disruption_report.txt",
)

print("[OK] Task 5: Executive report     (context: ALL tasks)")
print()
print("Task dependency chain:")
print("  task_monitor")
print("    +-> task_route ---------> task_compliance")
print("    +-> task_route ---+")
print("    +------------------+-> task_comms")
print("  ALL -----------------> task_report -> file output")

## Part 3 - Assemble the Crew and Run (30 min)

In [ ]:
# Cell 15 - Assemble the Crew
from crewai import Crew, Process

globalflow_crew = Crew(
    agents=[
        disruption_monitor,
        route_optimiser,
        supplier_comms,
        compliance_officer,
        report_writer,
    ],
    tasks=[
        task_monitor,
        task_route,
        task_comms,
        task_compliance,
        task_report,
    ],
    process=Process.hierarchical,
    manager_llm=GROQ_MANAGER,   # Groq 70B orchestrates delegation
    verbose=True,
    memory=True,                # Enables long-term + short-term memory
    output_log_file="crew_run.log",
)

print("[OK] GlobalFlow Crew assembled")
print(f"  Agents:      {len(globalflow_crew.agents)}")
print(f"  Tasks:       {len(globalflow_crew.tasks)}")
print(f"  Process:     {globalflow_crew.process.value}")
print(f"  Manager LLM: {GROQ_MANAGER}")
print(f"  Memory:      {globalflow_crew.memory}")

In [ ]:
# Cell 16 - Trigger: simulate a Rotterdam port closure
trigger_input = {
    "disruption_alert": (
        "ALERT: Port of Rotterdam (GlobalFlow EU hub) has declared force majeure "
        "due to severe North Sea storm surge. Expected closure: 18-24 hours. "
        "340 containers from GlobalFlow clients are currently docked. "
        "12 Maersk vessels en-route have been diverted to Felixstowe. "
        "Incident started: 2025-06-18 06:30 UTC. "
        "Client SLA breach window opens in 6 hours."
    )
}

print("[ALERT] TRIGGERING GlobalFlow Disruption Response Crew")
print("=" * 60)
print(trigger_input["disruption_alert"])
print("=" * 60)
print()
print("Starting crew kickoff - expect 2-5 minutes on Groq free tier.")
print("Watch each agent Thought -> Action -> Observation loop below.")
print()

result = globalflow_crew.kickoff(inputs=trigger_input)

In [ ]:
# Cell 17 - Review the final output
print("\n" + "=" * 60)
print("FINAL EXECUTIVE BRIEFING")
print("=" * 60)
print(result.raw)
print()
print("-" * 60)
print(f"Token usage: {result.token_usage}")

In [ ]:
# Cell 18 - Inspect the saved report file
import os

report_file = "globalflow_disruption_report.txt"
if os.path.exists(report_file):
    size = os.path.getsize(report_file)
    print(f"[OK] Report saved: '{report_file}'  ({size} bytes)")
    print()
    with open(report_file, "r") as f:
        print(f.read())
else:
    print("[WARN] Report file not found - check verbose output above.")
    print("Falling back to result.raw:")
    print(result.raw)

In [ ]:
# Cell 19 - Inspect crew memory (long-term)
import glob

memory_files = glob.glob("*.db") + glob.glob(".crewai/**/*.db", recursive=True)
if memory_files:
    print("Memory database files found:")
    for f in memory_files:
        size = os.path.getsize(f)
        print(f"  {f}  ({size:,} bytes)")
    print()
    print("[TIP] Re-run Cell 16 - agents will reference previous disruption context.")
else:
    print("No memory DB found yet. Run the crew kickoff first (Cell 16).")
    print("Memory DB appears after first successful run.")

In [ ]:
# Cell 20 - Token usage and rough cost estimate
print("Token Usage Summary")
print("=" * 40)
if hasattr(result, 'token_usage') and result.token_usage:
    tu = result.token_usage
    print(f"  Prompt tokens:     {tu.prompt_tokens:>8,}")
    print(f"  Completion tokens: {tu.completion_tokens:>8,}")
    print(f"  Total tokens:      {tu.total_tokens:>8,}")
    # Groq llama-3.3-70b pricing (approx): $0.59/1M input, $0.79/1M output
    input_cost  = (tu.prompt_tokens     / 1_000_000) * 0.59
    output_cost = (tu.completion_tokens / 1_000_000) * 0.79
    print(f"  Est. cost (70B):   ${input_cost + output_cost:.5f}")
else:
    print("  Token usage not available for this run.")

## Extension Tasks

Work through these after the core lab. Estimated time: 30-60 min.

---

### Extension 1 - Add a Financial Analyst Agent

Create a 6th agent that calculates total EUR exposure from the disruption:

```python
financial_analyst = Agent(
    role="Supply Chain Financial Analyst",
    goal=(
        "Calculate total EUR exposure: rerouting cost delta, "
        "SLA penalty clauses triggered, insurance deductible, "
        "and opportunity cost of delayed deliveries."
    ),
    backstory=(
        "CFA-qualified financial analyst specialising in logistics cost modelling. "
        "Always presents base case, worst case, and best case scenarios."
    ),
    llm=GROQ_SMART,
    verbose=True,
    max_iter=3,
)

task_financial = Task(
    description="Calculate total EUR exposure: rerouting, SLA penalties, insurance.",
    expected_output="Financial exposure table: base / worst / best case in EUR.",
    agent=financial_analyst,
    context=[task_monitor, task_route],
)
```

Add `financial_analyst` to `agents=` and `task_financial` to `tasks=` in the Crew, then re-run.

---

### Extension 2 - Parallel Async Execution

`task_comms` and `task_compliance` are independent and can run in parallel:

```python
task_comms = Task(..., async_execution=True)
task_compliance = Task(..., async_execution=True)
```

Use `crew.kickoff_async()` and compare wall-clock time vs sequential.

---

### Extension 3 - Human-in-the-Loop Gate

Add a human approval step before the report is written:

```python
task_report = Task(..., human_input=True)
```

Re-run - a prompt will appear asking you to approve before the report is written.

---

### Extension 4 - Switch Groq Model Tiers

Try assigning different models per agent based on task complexity:

```python
GROQ_FAST  = "groq/llama-3.1-8b-instant"    # supplier_comms (drafting)
GROQ_SMART = "groq/llama-3.3-70b-versatile" # monitor, router, compliance
```

Compare output quality vs token cost across tiers.

## Extension 1 — Financial Analyst Agent (EUR Exposure Calculator)

> **Goal:** Add a 6th agent that quantifies the full EUR financial exposure from the disruption: rerouting cost delta, SLA penalty clauses triggered, insurance deductible, and opportunity cost of delayed deliveries.  
> **Time estimate:** ~25 min  

### What we add
| # | Component | Purpose |
|---|-----------|--------|
| E1-A | `financial_analyst` Agent | CFA-level cost modeller |
| E1-B | `task_financial` Task | EUR exposure calculation (3-scenario table) |
| E1-C | Crew v2 (`globalflow_crew_v2`) | Original 5 agents + Financial Analyst |
| E1-D | Extended trigger run | Full 6-agent crew kickoff |
| E1-E | Report comparison | Compare original vs extended output |

---

In [ ]:
# Cell E1-A — Financial Analyst Agent (Extension 1)
# ─────────────────────────────────────────────────────────────────────────
# WHY a 6th agent?
#   The original 5-agent crew detects disruptions, reroutes, comms suppliers,
#   checks compliance, and writes a report — but never quantifies EUR exposure.
#   Executives need a number. This agent provides it.
#
# WHY GROQ_SMART (70B)?
#   Financial modelling needs multi-step arithmetic reasoning and structured
#   table output. The 8B model lacks reliable chain-of-thought for this.
#
# WHY max_iter=4?
#   Three scenario calculations (base / worst / best) + one summary step
#   = 4 reasoning iterations is the safe upper bound on Groq free tier.

from crewai import Agent

financial_analyst = Agent(
    role="Supply Chain Financial Analyst",
    goal=(
        "Calculate the total EUR financial exposure from the logistics disruption. "
        "Model three scenarios: base case, worst case, and best case. "
        "Cover: (1) rerouting cost delta vs standard lane, "
        "(2) SLA penalty clauses triggered per contract tier, "
        "(3) cargo insurance deductible for declared force majeure, "
        "(4) opportunity cost of delayed deliveries (perishables priority). "
        "Always express final exposure in EUR with confidence intervals."
    ),
    backstory=(
        "You are a CFA-qualified financial analyst with 10 years specialising in "
        "logistics cost modelling at Kuehne+Nagel and DB Schenker. "
        "You have built Monte Carlo models for port disruption scenarios, know every "
        "SLA penalty clause by heart, and understand cargo insurance clauses better "
        "than most underwriters. You speak in numbers, not narratives. "
        "Your outputs always include a base / worst / best case table with "
        "clearly labelled assumptions. You never present a single-point estimate "
        "without confidence bounds."
    ),
    llm=GROQ_SMART,          # 70B model - financial reasoning needs it
    verbose=True,
    max_iter=4,
    # No tools needed: works entirely from disruption + route context
)

print("[OK] Extension 1 — Agent 6 created:", financial_analyst.role)
print(f"     LLM: {GROQ_SMART}")
print(f"     Max iterations: {financial_analyst.max_iter}")


In [ ]:
# Cell E1-B — task_financial (Extension 1)
# ─────────────────────────────────────────────────────────────────────────
# CONTEXT INJECTION:
#   context=[task_monitor, task_route] injects:
#     • task_monitor  -> severity, shipment count, duration estimate
#     • task_route    -> cost delta %, delay hours, CO2 delta per route
#   This is the minimum needed for financial modelling.
#   We do NOT inject task_compliance to keep the context focused.
#
# COST ASSUMPTIONS (injected in description so the agent has a baseline):
#   These are realistic GlobalFlow-scale numbers. The agent is instructed to
#   use them unless the context provides better figures.
#
# OUTPUT FILE:
#   Separate file so it can be attached to finance systems independently.

from crewai import Task

COST_ASSUMPTIONS = """
GlobalFlow cost assumptions (use unless context overrides):
  - Standard Rotterdam lane cost per container: EUR 1,850
  - Containers affected: 340
  - SLA tier A penalty (>6h delay): EUR 500/container (80 containers)
  - SLA tier B penalty (>12h delay): EUR 1,200/container (40 containers)
  - Cargo insurance deductible (force majeure clause): EUR 250,000 flat
  - Perishables opportunity cost: EUR 8,000/hour (estimated 22 containers)
  - Staff overtime + emergency ops: EUR 45,000 flat estimate
"""

task_financial = Task(
    description=(
        "Using the disruption report and route optimisation output in your context, "
        "calculate the total EUR financial exposure for GlobalFlow. "
        f"{COST_ASSUMPTIONS}"
        "Produce a structured financial exposure table with these exact rows:\n"
        "  1. Rerouting cost delta (vs standard lane)\n"
        "  2. SLA penalties triggered (Tier A + Tier B)\n"
        "  3. Cargo insurance deductible\n"
        "  4. Perishables opportunity cost\n"
        "  5. Staff / emergency ops overhead\n"
        "  6. TOTAL EXPOSURE\n"
        "Show three columns: BASE CASE | WORST CASE | BEST CASE.\n"
        "Add a one-paragraph CFO summary below the table.\n"
        "Save full output to 'globalflow_financial_exposure.txt'."
    ),
    expected_output=(
        "A 6-row EUR exposure table (Base / Worst / Best) covering rerouting, "
        "SLA penalties, insurance, perishables, overhead, and total. "
        "Plus a CFO summary paragraph. Saved to 'globalflow_financial_exposure.txt'."
    ),
    agent=financial_analyst,
    context=[task_monitor, task_route],  # cost and delay data feeds in here
    output_file="globalflow_financial_exposure.txt",
)

print("[OK] Extension 1 — task_financial created")
print("     Context dependencies: task_monitor, task_route")
print("     Output file: globalflow_financial_exposure.txt")


In [ ]:
# Cell E1-C — Assemble Extended Crew v2 (Extension 1)
# ─────────────────────────────────────────────────────────────────────────
# CHANGES vs original crew:
#   + financial_analyst added to agents list (position 5, before report_writer)
#   + task_financial added to tasks list (after task_compliance, before task_report)
#   + task_report context updated to include task_financial output
#
# WHY process=Process.hierarchical is kept?
#   Hierarchical process lets the manager LLM decide task ordering at runtime.
#   task_financial is injected into task_report via context, so the manager
#   automatically schedules financial_analyst before report_writer.
#
# NOTE: task_report is redefined here with the extended context.
#       The original task_report (Cell 14) remains intact for comparison.

from crewai import Crew, Process, Task

# Redefine task_report with financial context included
task_report_v2 = Task(
    description=(
        "Compile ALL outputs — disruption intel, route options, supplier actions, "
        "compliance status, AND financial exposure — into a single executive briefing.\n"
        "Use these exact headings:\n"
        "  SITUATION: what happened and severity\n"
        "  IMPACT: shipments affected, EUR cost exposure (use financial analyst numbers)\n"
        "  RESPONSE: chosen re-route, supplier actions, compliance status\n"
        "  FINANCIAL SUMMARY: base / worst / best EUR totals from financial analyst\n"
        "  NEXT STEPS: exactly 3 numbered actions with owners and deadlines\n"
        "Maximum 500 words (increased from 400 to accommodate financial section). "
        "Save to 'globalflow_disruption_report_v2.txt'."
    ),
    expected_output=(
        "Complete executive briefing saved to 'globalflow_disruption_report_v2.txt', "
        "5-section structure including FINANCIAL SUMMARY, maximum 500 words."
    ),
    agent=report_writer,
    context=[task_monitor, task_route, task_comms, task_compliance, task_financial],
    output_file="globalflow_disruption_report_v2.txt",
)

# Build the extended 6-agent crew
globalflow_crew_v2 = Crew(
    agents=[
        disruption_monitor,
        route_optimiser,
        supplier_comms,
        compliance_officer,
        financial_analyst,   # <-- NEW: position 5
        report_writer,
    ],
    tasks=[
        task_monitor,
        task_route,
        task_comms,
        task_compliance,
        task_financial,      # <-- NEW: position 5
        task_report_v2,      # <-- UPDATED: includes financial context
    ],
    process=Process.hierarchical,
    manager_llm=GROQ_MANAGER,
    verbose=True,
    memory=True,
    output_log_file="crew_run_v2.log",
)

print("[OK] Extension 1 — GlobalFlow Crew v2 assembled")
print(f"  Agents:      {len(globalflow_crew_v2.agents)} (was 5)")
print(f"  Tasks:       {len(globalflow_crew_v2.tasks)} (was 5)")
print(f"  Process:     {globalflow_crew_v2.process.value}")
print(f"  Manager LLM: {GROQ_MANAGER}")
print()
print("Updated task dependency graph:")
print("  task_monitor")
print("    +-> task_route ─────────────────> task_compliance")
print("    +-> task_route ──+               ")
print("    +────────────────+──> task_comms ")
print("    +-> task_route ──────────────────> task_financial  [NEW]")
print("  ALL ─────────────────────────────────> task_report_v2 -> file output")


In [ ]:
# Cell E1-D — Run the Extended Crew (Extension 1 kickoff)
# ─────────────────────────────────────────────────────────────────────────
# Same trigger_input as the original run so outputs are directly comparable.
# The financial_analyst will pick up route cost deltas from task_route context
# and model them against the cost assumptions embedded in task_financial.

print("[ALERT] TRIGGERING GlobalFlow Disruption Response Crew v2 (6 agents)")
print("=" * 60)
print(trigger_input["disruption_alert"])
print("=" * 60)
print()
print("New this run: Financial Analyst Agent will calculate EUR exposure.")
print("Expected additional output: globalflow_financial_exposure.txt")
print()
print("Starting crew v2 kickoff — expect 3-7 minutes on Groq free tier.")
print()

result_v2 = globalflow_crew_v2.kickoff(inputs=trigger_input)


In [ ]:
# Cell E1-E — Review Extended Outputs + Compare (Extension 1)
# ─────────────────────────────────────────────────────────────────────────

import os

print("\n" + "=" * 60)
print("EXTENSION 1 — FINANCIAL EXPOSURE REPORT")
print("=" * 60)

fin_file = "globalflow_financial_exposure.txt"
if os.path.exists(fin_file):
    with open(fin_file) as f:
        print(f.read())
else:
    print("[WARN] Financial exposure file not found. Check verbose output above.")
    # Fallback: extract financial section from the v2 executive report
    print("Falling back to result_v2.raw (financial section should be inside):")
    print(result_v2.raw)

print()
print("=" * 60)
print("EXTENSION 1 — EXECUTIVE BRIEFING v2 (WITH FINANCIAL SUMMARY)")
print("=" * 60)

rep_file = "globalflow_disruption_report_v2.txt"
if os.path.exists(rep_file):
    with open(rep_file) as f:
        print(f.read())
else:
    print(result_v2.raw)

print()
print("─" * 60)
print("COMPARISON: Original (5 agents) vs Extended (6 agents)")
print("─" * 60)
orig = open('globalflow_disruption_report.txt').read() if os.path.exists('globalflow_disruption_report.txt') else "(original report not found - run Part 3 cells first)"
v2   = open(rep_file).read() if os.path.exists(rep_file) else result_v2.raw

print(f"Original report word count:  {len(orig.split()):>5}")
print(f"Extended report word count:  {len(v2.split()):>5}")
print(f"Financial file word count:   ", end="")
if os.path.exists(fin_file):
    print(f"{len(open(fin_file).read().split()):>5}")
else:
    print("  N/A")

print()
print("Token comparison:")
print(f"  Original run tokens: {result.token_usage.total_tokens:>8,}" if hasattr(result, 'token_usage') and result.token_usage else "  Original run tokens: not available")
print(f"  Extended run tokens: {result_v2.token_usage.total_tokens:>8,}" if hasattr(result_v2, 'token_usage') and result_v2.token_usage else "  Extended run tokens: not available")


In [ ]:
# Cell E1-F — Token Usage + Cost for Extended Crew (Extension 1)
# ─────────────────────────────────────────────────────────────────────────

print("Token Usage Summary — Extended Crew v2 (6 agents)")
print("=" * 50)
if hasattr(result_v2, 'token_usage') and result_v2.token_usage:
    tu = result_v2.token_usage
    print(f"  Prompt tokens:     {tu.prompt_tokens:>8,}")
    print(f"  Completion tokens: {tu.completion_tokens:>8,}")
    print(f"  Total tokens:      {tu.total_tokens:>8,}")
    input_cost  = (tu.prompt_tokens     / 1_000_000) * 0.59
    output_cost = (tu.completion_tokens / 1_000_000) * 0.79
    total_cost  = input_cost + output_cost
    print(f"  Est. cost (70B):   ${total_cost:.5f}")
    print()
    if hasattr(result, 'token_usage') and result.token_usage:
        orig_total = result.token_usage.total_tokens
        delta      = tu.total_tokens - orig_total
        pct        = (delta / orig_total) * 100
        print(f"  Delta vs original: +{delta:,} tokens ({pct:.1f}% overhead for financial analysis)")
else:
    print("  Token usage not available for this run.")

print()
print("[DONE] Extension 1 complete.")
print("Key files produced:")
print("  globalflow_financial_exposure.txt    <- EUR exposure (base/worst/best)")
print("  globalflow_disruption_report_v2.txt  <- Executive briefing with finance")
print("  crew_run_v2.log                      <- Full verbose agent log")
